# Function 6: 5D Composite Constraint Optimisation
**Programme:** Imperial College London – Professional Certificate in Machine Learning & AI  
**Domain Context:** Dairy Total Mixed Ration (TMR) Diet Formulation (5 Feed Proportions)  
**Author:** Dr. Joseph Neary  

---

## 1. Problem Framing & Objective
Function 6 accepts a 5D feature vector $\mathbf{x} = (x_1, x_2, x_3, x_4, x_5)$ bounded within $[0, 1]^5$. The objective is to **maximise** a composite score $y$ (where negative penalties are structured such that optimal performance pushes the output as close to zero as possible).

We build a **Gaussian Process (GP) Surrogate Model** using an RBF kernel with target normalization (`normalize_y=True`) and noise regularization ($\alpha = 10^{-3}$). We evaluate candidate utility across a $15 \times 15 \times 15 \times 15 \times 15$ grid ($759,375$ points) using the **Upper Confidence Bound (UCB)** acquisition function.

## 2. Ingestion & Baseline Data Analysis
We ingest the $N = 10$ seed observations provided in `Data/Raw/function_6/` and display them sorted by descending target output $y$.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

# ==========================================
# 1. LOAD DATA FROM YOUR RAW DATA DIRECTORY
# ==========================================
base_dir = r"C:\Projects\ML\capstone\Imperial-ML-Capstone-Project"

input_path = os.path.join(base_dir, 'Data', 'Raw', 'function_6', 'initial_inputs.npy')
output_path = os.path.join(base_dir, 'Data', 'Raw', 'function_6', 'initial_outputs.npy')

X_init = np.load(input_path)   # Shape: (10, 5) - 5D input coordinates
y_init = np.load(output_path)  # Shape: (10,)   - Target score

# Display initial data table sorted by current peak score
df_f6 = pd.DataFrame(X_init, columns=['x1', 'x2', 'x3', 'x4', 'x5'])
df_f6['y'] = y_init

print("=== Function 6: Initial Observations (Sorted High to Low) ===")
print(df_f6.sort_values(by='y', ascending=False).reset_index(drop=True))
print("\n" + "="*60 + "\n")

=== Function 6: Initial Observations (Sorted High to Low) ===
          x1        x2        x3        x4        x5         y
0   0.728186  0.154693  0.732552  0.693997  0.056401 -0.714265
1   0.618812  0.331802  0.187288  0.756238  0.328835 -0.829237
2   0.782880  0.536336  0.443284  0.859700  0.010326 -0.935757
3   0.536797  0.308781  0.411879  0.388225  0.522528 -1.144785
4   0.242384  0.844100  0.577809  0.679021  0.501953 -1.209955
5   0.145111  0.896685  0.896322  0.726272  0.236272 -1.233786
6   0.784958  0.910682  0.708120  0.959225  0.004911 -1.247049
7   0.432166  0.715618  0.341819  0.705000  0.614962 -1.294247
8   0.757594  0.355831  0.016523  0.434207  0.112433 -1.309116
9   0.021735  0.428084  0.835939  0.489489  0.511082 -1.356682
10  0.770620  0.114404  0.046780  0.648324  0.273549 -1.536058
11  0.629308  0.803484  0.811408  0.045613  0.110624 -1.622839
12  0.729523  0.748106  0.679775  0.356552  0.671054 -1.672200
13  0.945069  0.288459  0.978806  0.961656  0.598016 -1.

## 3. GP Surrogate Fitting & 5D Grid Evaluation
We fit a non-parametric GP regressor with a composite RBF kernel:

$$k(\mathbf{x}, \mathbf{x}') = C \cdot \exp\left(-\frac{\|\mathbf{x} - \mathbf{x}'\|^2}{2\ell^2}\right)$$

We evaluate the **Upper Confidence Bound (UCB)** acquisition utility across the 5D lattice:

$$\text{UCB}(\mathbf{x}) = \mu(\mathbf{x}) + \kappa \cdot \sigma(\mathbf{x})$$

Where $\kappa = 2.0$ balances exploitation of high-performing diet configurations ($\mu$) with unmapped nutritional space exploration ($\sigma$).

In [2]:
# ==========================================
# 2. FIT GAUSSIAN PROCESS (GP) SURROGATE MODEL
# ==========================================
kernel = C(1.0, (1e-5, 1e5)) * RBF(length_scale=0.5, length_scale_bounds=(1e-3, 1e2))

gp = GaussianProcessRegressor(
    kernel=kernel, 
    n_restarts_optimizer=10, 
    alpha=1e-3,          # Regularization for noise tolerance
    normalize_y=True,    # Standardises target y values
    random_state=42
)

# Fit GP model to 5D inputs
gp.fit(X_init, y_init)

print("=== Fitted Kernel Hyperparameters ===")
print(gp.kernel_)
print("="*60 + "\n")

# ==========================================
# 3. 5D GRID SEARCH FOR UCB MAXIMISATION
# ==========================================
grid_res = 15  # 15^5 = 759,375 evaluation points
x_dom = np.linspace(0, 1, grid_res)

# Generate 5D meshgrid
X1, X2, X3, X4, X5 = np.meshgrid(x_dom, x_dom, x_dom, x_dom, x_dom)
X_test = np.column_stack([
    X1.ravel(), X2.ravel(), X3.ravel(), X4.ravel(), X5.ravel()
])

# Predict posterior mean and standard deviation across 5D space
mu, sigma = gp.predict(X_test, return_std=True)

# Upper Confidence Bound (UCB)
kappa = 2.0
ucb_values = mu + kappa * sigma

best_idx = np.argmax(ucb_values)
next_query = X_test[best_idx]

# ==========================================
# 4. PORTAL SUBMISSION FORMATTING
# ==========================================
formatted_coords = " - ".join([f"{coord:.6f}" for coord in next_query])

print("=== RECOMMENDED SUBMISSION FOR FUNCTION 6 (WEEK 1) ===")
print(f"Coordinates (x1 to x5): {list(next_query)}")
print(f"Portal Submission Format:  {formatted_coords}")
print("======================================================")

=== Fitted Kernel Hyperparameters ===
1.13**2 * RBF(length_scale=0.489)

=== RECOMMENDED SUBMISSION FOR FUNCTION 6 (WEEK 1) ===
Coordinates (x1 to x5): [np.float64(0.2857142857142857), np.float64(0.3571428571428571), np.float64(0.42857142857142855), np.float64(0.7142857142857142), np.float64(0.0)]
Portal Submission Format:  0.285714 - 0.357143 - 0.428571 - 0.714286 - 0.000000


## Imperial Capstone Reflection: Function 6 (Week 1)

**Query Input Coordinates:** `0.285714 - 0.357143 - 0.428571 - 0.714286 - 0.000000`

---

### 1. What method was used and why?
I implemented a **Gaussian Process (GP) Regression** surrogate model paired with an **Upper Confidence Bound (UCB)** acquisition function evaluated across a $15 \times 15 \times 15 \times 15 \times 15$ grid ($759,375$ candidate points) spanning the 5D unit hypercube $[0, 1]^5$.

A non-parametric GP surrogate model was selected because Function 6 represents a multi-variable composite score function where output penalties (flavor, consistency, cost, waste) are negative by design, requiring maximisation toward zero. We specified a composite Radial Basis Function (RBF) kernel:

$$k(\mathbf{x}, \mathbf{x}') = C \cdot \exp\left(-\frac{\|\mathbf{x} - \mathbf{x}'\|^2}{2\ell^2}\right)$$

Target normalization (`normalize_y=True`) and noise regularization ($\alpha = 10^{-3}$) were incorporated to stabilize hyperparameter estimation via Maximum Marginal Likelihood (MML). The fitted kernel converged to a variance scale of $C = 1.13^2$ and a length scale of $\ell = 0.489$.

---

### 2. Was the query focused on exploration or exploitation?
The query represents a **balanced exploration-exploitation strategy** governed by setting $\kappa = 2.0$ in the UCB acquisition utility formula:

$$\text{UCB}(\mathbf{x}) = \mu(\mathbf{x}) + \kappa \cdot \sigma(\mathbf{x})$$

In a 5D feature domain with only $N = 10$ seed observations, the volume of the search space creates extreme data sparsity. Pure exploitation ($\kappa = 0$) risks trapping the search in a sub-optimal local neighborhood. Setting $\kappa = 2.0$ mathematically weights high predicted performance ($\mu(\mathbf{x})$) while prioritizing candidate points in regions where epistemic uncertainty ($\sigma(\mathbf{x})$) remains high across the 5D lattice.

---

### 3. What did the baseline dataset teach us about the underlying function?
The fitted length scale ($\ell = 0.489$) indicates that the composite response surface maintains smooth spatial continuity across the 5D input domain. However, because 10 points in 5 dimensions leave large unmapped volumes, posterior uncertainty ($\sigma(\mathbf{x})$) dominates across most of the search hypercube, guiding the acquisition engine toward boundary interactions (such as $x_5 = 0.0$).

---

### 4. What is the strategy for the next round?
Upon receiving the true composite score output $y_{\text{new}}$ for `0.285714 - 0.357143 - 0.428571 - 0.714286 - 0.000000`, I will append the 11th evaluation point to $\mathbf{X}$ and $y$ and refit the 5D GP surrogate model:
* **If $y_{\text{new}}$ brings the total score closer to zero (improvement):** Shift toward local exploitation ($\kappa \to 1.0–1.2$) to refine the parameter proportions around this high-efficiency zone.
* **If $y_{\text{new}}$ degrades the score:** Maintain $\kappa = 2.0$ or evaluate switching acquisition functions to **Expected Improvement (EI)** to explore alternate sub-volumes of the 5D unit hypercube.